<a href="https://colab.research.google.com/github/sanjaliroy/berkeley-homes-wildfire-agent-simulation/blob/main/notebooks/wildfire_transcript_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homeowners Interview Transcript Pre-processing

Course: INFO 290: Fundamentals of Generative AI (Spring 2026)







Creator: Amrita Nambiar

What this notebook covers:
1.   Transcription Cleaning: Filters raw text to isolate homeowner responses.
2.   AI-Driven Extraction: Uses Claude claude-sonnet-4-5 model to transform cleaned transcripts into structured profiles.
3.  Data Export: Saves results to Google Drive as {agent_name}.yaml (profiles) and extraction_debug.jsonl (logs).

Why this matters:


1.   Agent Persona Creation: The extracted profiles form the basis for creating realistic and diverse agent personas in a wildfire mitigation simulation.
2.    Structured Data: Converts unstructured interview data into a structured format that can be easily consumed by simulation models.
3.    Automated Extraction: Leverages Generative AI to automate the extraction process, reducing manual effort and ensuring consistency.



## Install Dependencies

In [ ]:
!pip install python-docx anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 13.2 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata, drive
import anthropic
from pathlib import Path

import re
import json
import yaml
from docx import Document
from pathlib import Path
import pandas as pd

## Configuration

In [ ]:
# Google Drive folder with meeting notes
notes_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes"

# Output folder
output_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes/outputs"

# Output filenames
# Each agent is saved as <agent_id>.yaml in output_folder_path
debug_jsonl_filename = "notes_extraction_debug.jsonl"

# Model for extraction
model = "claude-sonnet-4-5"
max_tokens = 16000

print("Configuration completed")
print(f"Notes folder  : {notes_folder_path}")
print(f"Output folder : {output_folder_path}")
print(f"Model         : {model}")


Configuration completed
Notes folder  : /content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes
Output folder : /content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes/outputs
Model         : claude-sonnet-4-5


## Mount Google Drive & Load API Key

In [ ]:
# Mount Google Drive
drive.mount("/content/drive")
print("Google Drive mounted")

# Load API key
anthropic_api_key = userdata.get('wildfire_claude')

if anthropic_api_key:
    os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key
    print("API key loaded.")
else:
    print("API key not found. Please add ANTHROPIC_API_KEY to Colab secrets.")

# Verify notes folder exists
notes_dir = Path(notes_folder_path)
if not notes_dir.exists():
    print(f"\nNotes folder not found: {notes_folder_path}")
else:
    docx_files = sorted(notes_dir.glob("*.docx"))
    print(f"\nFound {len(docx_files)} .docx note file(s):")
    for f in docx_files:
        print(f"   • {f.name}")


Mounted at /content/drive
Google Drive mounted
API key loaded.

Found 9 .docx note file(s):
   • Candace.docx
   • Greg.docx
   • Heidi.docx
   • Katarina.docx
   • Matt.docx
   • Max.docx
   • Ruth.docx
   • Shah.docx
   • anonymous_1.docx


## Helper Functions

In [ ]:
# Extract plain text from .docx
def extract_text_from_docx(docx_path: Path) -> str:
    # Read all paragraphs from a .docx file and return as plain text
    doc = Document(str(docx_path))
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [ ]:
# Extract plain text from .docx
def extract_text_from_docx(docx_path: Path) -> str:
    # Read all paragraphs from a .docx file and return as plain text
    doc = Document(str(docx_path))
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)


In [ ]:
extraction_prompt = """\
You are a research assistant helping build realistic agent personas for a wildfire \
mitigation simulation set in the Berkeley Hills, California.

Below are researcher-written meeting notes from an interview with a Berkeley Hills \
resident who declined to be recorded. The notes are written in third person by the \
interviewer and summarise what the resident said — they are NOT a verbatim transcript. \
Direct quotes may be scarce or absent.

Your job is to extract a structured profile that will be used to initialise a \
simulation agent representing the resident's perspective and attitudes.

NOTES:
{transcript}


---


EXTRACTION ORDER — MANDATORY


Extract fields in this order. Each stage constrains the next.


STAGE 1 — held_out_responses
STAGE 2 — memory_seeds
STAGE 3 — seed_narrative


The boundary rule: for each of the five intervention types, identify the resident's \
single most representative attitude, belief, or concern as captured in the notes, and \
place it in held_out_responses. Everything else they conveyed about that event — \
actions taken, context, other feelings — belongs in memory_seeds. \
seed_narrative establishes identity and attitude only; it must not repeat episodic \
detail already in memory_seeds and does not reproduce any held-out content.\


---


GROUNDING RULE — MANDATORY FOR ALL STAGES


Every field must be grounded directly in the notes. This means:


- Do not invent facts, events, emotions, or attitudes the notes do not support.
- Because these are researcher summaries, the resident's voice is mediated. \
  Reconstruct attitudes in plausible first-person language, but stay within \
  what the notes explicitly state or clearly imply.
- Do not fill gaps with plausible-sounding detail. If something was not mentioned, \
  use null or omit it entirely.
- Before writing any field, ask: \"Can I point to a specific bullet or section in \
  the notes that supports this?\" If not, do not include it.


A shorter, accurate extraction is preferable to a fuller one that introduces invented content.


---


STAGE 1: held_out_responses


For each intervention type below, capture the resident's single most representative \
attitude or belief — what they thought, felt, or believed — as described in the notes. \
ONLY 1 RESPONSE PER INTERVENTION.\
Because verbatim quotes may not exist, write a close first-person reconstruction \
that faithfully reflects the note content. Mark reconstructions with [reconstructed].\
Use null if the topic was not discussed.


 (a) insurance_non_renewal
 (b) defensible_space_inspection
 (c) neighbor_pressure
 (d) firewise_outreach
 (e) resources_assistance


Once extracted, treat each held-out response as a quarantine item. \
Stages 2 and 3 must not reproduce it verbatim or as a close paraphrase.


---


STAGE 2: memory_seeds


Memory seeds are discrete episodic memories retrieved dynamically by the simulation. \
They should cover the full range of what the notes describe:


- Background context for intervention events
- Actions taken around those events
- Other feelings and attitudes expressed (EXCLUDING the held-out response)
- Conversations with neighbors, officials, or contractors mentioned in the notes
- Observations about the property, neighborhood, insurance, or fire risk
- Routine mitigation steps and efforts


Because these are summarised notes, seeds may be written in inferred first-person \
(\"I was told...\", \"I asked about...\") where the notes clearly support the inference. \
Do not invent detail beyond what is stated.


Include up to 20 seeds as the notes support (notes are shorter than full transcripts). \
Aim for a spread across the importance range. All seeds must be grounded in the notes.


---


STAGE 3: seed_narrative


The seed narrative is the agent's stable identity. It must be 10 sentences maximum.


Before writing each sentence, ask:
 1. \"Does this reproduce the held-out response verbatim or as a close paraphrase?\" If yes, omit it.
 2. \"Does this duplicate a memory seed?\" If yes, omit.
 3. \"Is this grounded in the notes?\" If no, omit.


It must describe behaviours and styles, not specific events.
Structure in second person ('You are...'):


 (1) Who you are and your living situation.
 (2) Your fire risk awareness and attitude.
 (3) Your general approach to tasks and community.


Weave these into natural prose without subheadings.
If the notes contain contradictions, reflect the tension honestly.


---


Extract the following and respond ONLY with valid JSON (no markdown, no preamble, \
no trailing text):


{{
 \"agent_id\": \"<firstname_lowercase or anonymous_N if name unknown>\",
 \"display_name\": \"<privacy-preserving pseudonym matching the person's approximate age, \
gender, and cultural background. For example: Patricia, James, Eleanor, Thomas, Vivian,
Carlos, Dorothy, Andy, Beatrice, Jerome, Nora, Felix, Harriet, Omar, Celeste. Never use the interviewee's real name.>",
 \"persona\": \"<3-5 sentence paragraph in third person capturing: who they are, how long \
they have lived there, their attitude toward fire risk, their relationship with the Ember \
ordinance and defensible space requirements, their trust in government and fire \
institutions, and any distinctive personality traits revealed in the notes>\",
 \"risk_zone\": \"<high | medium | low — ridge/hilltop=high, mid-hill=medium, flatter/lower=low, or null>\",
 \"compliance_status\": \"<compliant | partially_compliant | non_compliant | null>\",
 \"insurance_status\": \"<retained | non_renewed | dropped | null>\",
 \"held_out_responses\": {{
   \"insurance_non_renewal\": {{
     \"real_response\": \"<resident's most representative belief or attitude. \
First-person reconstruction if no direct quote; mark with [reconstructed]. Null if not discussed.>\",
     \"context\": \"<context from the notes, or null>\"
   }},
   \"defensible_space_inspection\": {{
     \"real_response\": \"<first-person reconstruction or null>\",
     \"context\": \"<context or null>\"
   }},
   \"neighbor_pressure\": {{
     \"real_response\": \"<first-person reconstruction or null>\",
     \"context\": \"<context or null>\"
   }},
   \"firewise_outreach\": {{
     \"real_response\": \"<first-person reconstruction or null>\",
     \"context\": \"<context or null>\"
   }},
   \"resources_assistance\": {{
     \"real_response\": \"<first-person reconstruction or null>\",
     \"context\": \"<context or null>\"
   }}
 }},
 \"memory_seeds\": [
   {{
     \"description\": \"<specific first-person memory grounded in the notes — past tense, \
concrete detail. May be inferred from third-person summary; do not reproduce held-out response.>\",
     \"importance\": \"<integer 1-10>\",
     \"type\": \"<observation | reflection | conversation>\",
     \"source\": \"<direct_experience | secondhand_community | secondhand_media | institutional>\"
   }}
 ],
 \"seed_narrative\": \"<Narrative in second person ('You are {{display_name}}...') \
structured across the dimensions described in Stage 3. Establish identity and attitude only. \
Do not reproduce held-out responses or memory seed detail. Grounded entirely in the notes. \
End verbatim with: 'You will now respond to situations as {{display_name}}. Focus only on \
what you would actually do in direct response to the situation presented. Do not add \
unrelated action items or narrative conclusions. Stay practical, not emotional or literary. \
Stay in their voice. Respond in first person only. Do not narrate actions in third person, \
do not use stage directions or asterisks, do not use headers, bold text, horizontal rules, \
em-dashes, or formatted sections. Write in plain short sentences. Keep your response concise.'>\",
 \"key_concerns\": [
   \"<specific worry or tension captured in the notes>\"
 ]
}}
{avoid_names_instruction}
"""

In [ ]:
# Call Claude to extract profile
def extract_agent_profile(client: anthropic.Anthropic, transcript: str, name: str, avoid_names_list: list = None) -> tuple:
    avoid_names_instruction = ""
    if avoid_names_list:
        avoid_names_str = ", ".join([f"'{n}'" for n in avoid_names_list])
        avoid_names_instruction = f"CRITICAL: The 'display_name' you generate MUST NOT be one of the following: {avoid_names_str}. If your initial choice for 'display_name' is on this forbidden list, immediately generate a *different*, unique privacy-preserving pseudonym that still matches the persona, but is absolutely distinct from the forbidden names. Prioritize generating a new, unique name over strict adherence to the persona if a name clash is unavoidable with the original persona-matching name."

    prompt = extraction_prompt.format(transcript=transcript, avoid_names_instruction=avoid_names_instruction)

    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = message.content[0].text.strip()

    # Strip markdown fences if the model adds them despite instructions
    raw_text = re.sub(r'^```(?:json)?\s*', '', raw_text)
    raw_text = re.sub(r'\s*```$', '', raw_text).strip()

    try:
        profile = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {name}: {e}")
        profile = {"agent_id": name.lower(), "parse_error": str(e), "raw": raw_text}

    return profile, raw_text

In [ ]:
# Convert profile to agents.yaml format
def profile_to_agent_entry(profile: dict) -> dict:
    # Reshape extracted profile into the agents.yaml schema
    if "parse_error" in profile:
        return {"id": profile.get("agent_id", "unknown"), "error": profile["parse_error"]}

    return {
        "id":                      profile.get("agent_id"),
        "display_name":            profile.get("display_name"),
        "persona":                 profile.get("persona", ""),
        "risk_zone":               profile.get("risk_zone"),
        "compliance_status":       profile.get("compliance_status"),
        "insurance_status":        profile.get("insurance_status"),
        "held_out_responses":      profile.get("held_out_responses", {}),
        "memory_seeds":            profile.get("memory_seeds", []),
        "seed_narrative":          profile.get("seed_narrative", ""),
        "key_concerns":            profile.get("key_concerns", []),
    }


print("Helper functions defined")


Helper functions defined


## Run Extraction (All Transcripts)

In [ ]:
# Setup
claude_client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY from environment

notes_dir = Path(notes_folder_path)
output_dir = Path(output_folder_path)
output_dir.mkdir(parents=True, exist_ok=True)

if not notes_dir.exists():
    print(f"Notes folder not found: {notes_folder_path}")
else:
    docx_files = sorted(notes_dir.glob("*.docx"))
    print(f"Found {len(docx_files)} note file(s)")

print(f"Processing {len(docx_files)} note(s)...\n")
print("=" * 60)

# Pre-defined list of fallback names for guaranteed uniqueness if LLM generates duplicates
# These are additional names to ensure we don't run out of unique pseudonyms.
fallback_display_names_pool = [
    "Patricia", "James", "Eleanor", "Thomas", "Vivian", "Carlos", "Dorothy", "Andy",
    "Beatrice", "Jerome", "Nora", "Felix", "Harriet", "Omar", "Celeste", "Arthur",
    "Brenda", "Charles", "Diana", "Edward", "Florence", "George", "Helen", "Ivan",
    "Joyce", "Kenneth", "Linda", "Mark", "Nancy", "Oscar", "Pamela", "Richard",
    "Susan", "Timothy", "Ursula", "Victor", "Wendy", "Xavier", "Yvonne", "Zackary",
    "Audrey", "Bruce", "Catherine", "Donald", "Eve", "Frank", "Grace", "Harold",
    "Irene", "Jerry", "Karen", "Louis", "Maria", "Nathan", "Olive", "Peter"
]
used_fallback_names = set() # To track which fallback names have been used

agents = []
debug_entries = []
errors = []

seen_display_names = set() # Track all unique display names, whether LLM-generated or fallback

# Main extraction loop
for i, docx_path in enumerate(docx_files, 1):
    # Derive a short name from the filename
    stem = docx_path.stem
    agent_id_from_filename = stem.split(" - ")[0].split("_")[0].strip().lower()

    print(f"[{i}/{len(docx_files)}] {agent_id_from_filename}  ({docx_path.name})")

    try:
        # Extract raw text from notes .docx
        notes_text = extract_text_from_docx(docx_path)
        word_count = len(notes_text.split())
        print(f"         {word_count:,} words extracted")

        profile = None
        raw_response = None
        agent_entry = None

        # Retry mechanism for generating unique display names
        max_llm_retries = 3
        current_agent_tried_names = [] # Names tried by LLM for *this specific agent* that were duplicates

        final_display_name = None

        for retry_attempt in range(max_llm_retries + 1): # +1 for the initial attempt
            # Create a list of all names to avoid for the LLM call
            avoid_names_for_llm = list(seen_display_names.union(current_agent_tried_names))

            profile, raw_response = extract_agent_profile(claude_client, notes_text, agent_id_from_filename, avoid_names_list=avoid_names_for_llm)
            agent_entry = profile_to_agent_entry(profile)
            generated_display_name = agent_entry.get("display_name")

            if generated_display_name and generated_display_name not in seen_display_names:
                # Name is unique and hasn't been used, success!
                final_display_name = generated_display_name
                seen_display_names.add(final_display_name)
                break # Exit retry loop
            else:
                if retry_attempt < max_llm_retries:
                    print(f"         Warning: LLM generated duplicate display name '{generated_display_name}' (retry {retry_attempt+1}/{max_llm_retries}). Requesting a different name...")
                    if generated_display_name: # Only add if a name was actually generated
                        current_agent_tried_names.append(generated_display_name)
                else:
                    print(f"         Warning: LLM failed to generate a unique display name after {max_llm_retries} retries for {agent_id_from_filename}. Using a fallback name.")
                    # Fallback if LLM repeatedly generates duplicates
                    fallback_name_found = False
                    for fallback_name in fallback_display_names_pool:
                        if fallback_name not in seen_display_names:
                            final_display_name = fallback_name
                            seen_display_names.add(final_display_name)
                            used_fallback_names.add(final_display_name)
                            fallback_name_found = True
                            break
                    if not fallback_name_found:
                        raise ValueError("Exhausted all fallback display names. Cannot ensure uniqueness for all agents.")
                    break # Exit retry loop after fallback

        if final_display_name:
            agent_entry["display_name"] = final_display_name
            profile["display_name"] = final_display_name # Update profile for debug_entries as well
        else:
            # This should ideally not be reached if either LLM or fallback worked
            raise Exception("Failed to determine a unique display name for agent.")

        agents.append(agent_entry)

        # --- Save individual YAML file named by agent_id ---
        agent_id = agent_entry.get("id") or agent_id_from_filename
        agent_yaml_path = output_dir / f"{agent_id}.yaml"
        with open(agent_yaml_path, "w", encoding="utf-8") as f:
            yaml.dump(
                agent_entry,
                f,
                default_flow_style=False,
                allow_unicode=True,
                sort_keys=False,
            )
        print(f"         Saved → {agent_yaml_path.name}")

        # Save debug info
        debug_entries.append({
            "file": docx_path.name,
            "name": agent_id_from_filename,
            "word_count": word_count,
            "raw_llm_response": raw_response,
            "parsed_profile": profile
        })

        # Print summary row
        print(f"         agent_id={agent_entry.get('id')} | "
              f"risk={agent_entry.get('risk_zone')} | "
              f"compliance={agent_entry.get('compliance_status')}")

    except Exception as e:
        print(f"         ERROR: {e}")
        errors.append({"file": docx_path.name, "error": str(e)})

    print()

print("=" * 60)
print(f"Done. {len(agents)} agents extracted, {len(errors)} errors.")

Found 9 note file(s)
Processing 9 note(s)...

[1/9] candace  (Candace.docx)
         167 words extracted
         Saved → candace.yaml
         agent_id=candace | risk=high | compliance=compliant

[2/9] greg  (Greg.docx)
         1,059 words extracted
         Saved → greg.yaml
         agent_id=greg | risk=None | compliance=None

[3/9] heidi  (Heidi.docx)
         334 words extracted
         Saved → heidi.yaml
         agent_id=heidi | risk=high | compliance=partially_compliant

[4/9] katarina  (Katarina.docx)
         502 words extracted
         Saved → katarina.yaml
         agent_id=katarina | risk=high | compliance=compliant

[5/9] matt  (Matt.docx)
         354 words extracted
         Saved → matt.yaml
         agent_id=matt | risk=None | compliance=None

[6/9] max  (Max.docx)
         425 words extracted
         Saved → anonymous_1.yaml
         agent_id=anonymous_1 | risk=low | compliance=None

[7/9] ruth  (Ruth.docx)
         462 words extracted
         Saved → ruth.yaml


## Save Outputs to Google Drive

In [ ]:
# Save extraction_debug.jsonl
debug_jsonl_path = output_dir / debug_jsonl_filename

with open(debug_jsonl_path, "w", encoding="utf-8") as f:
    for entry in debug_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved debug log → {debug_jsonl_path}")
print(f"\nPer-agent YAML files saved to: {output_dir}")
for a in agents:
    aid = a.get('id', 'unknown')
    print(f"   • {aid}.yaml")

if errors:
    print(f"\n{len(errors)} file(s) failed:")
    for e in errors:
        print(f"   • {e['file']}: {e['error']}")


Saved debug log → /content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes/outputs/notes_extraction_debug.jsonl

Per-agent YAML files saved to: /content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes/outputs
   • candace.yaml
   • greg.yaml
   • heidi.yaml
   • katarina.yaml
   • matt.yaml
   • anonymous_1.yaml
   • ruth.yaml
   • mrs_shah.yaml
   • anonymous_1.yaml


Summary Table

In [ ]:
import pandas as pd

rows = []
for a in agents:
    rows.append({
        "ID":                      a.get("id", None),
        "Display Name":            a.get("display_name", None),
        "Persona":                 a.get("persona", None),
        "Risk Zone":               a.get("risk_zone"),
        "Compliance Status":       a.get("compliance_status"),
        "Insurance Status":        a.get("insurance_status"),
        "Held Out Responses":      a.get("held_out_responses", {}),
        "Memory Seeds":            a.get("memory_seeds", []),
        "Seed Narrative":          a.get("seed_narrative"),
        "Key Concerns":            a.get("key_concerns", []),
    })

df = pd.DataFrame(rows)
print("\nEXTRACTED AGENT SUMMARY")
display(df)

# Distribution checks — useful for ensuring variance across agent pool
print("\nDISTRIBUTIONS")
for col in ["ID", "Display Name", "Persona", "Risk Zone",
            "Compliance Status", "Insurance Status",
             "Held Out Responses","Memory Seeds","Seed Narrative","Key Concerns"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())


EXTRACTED AGENT SUMMARY


,ID,Display Name,Persona,Risk Zone,Compliance Status,Insurance Status,Held Out Responses,Memory Seeds,Seed Narrative,Key Concerns
0,candace,Harriet,Harriet is a Grizzly Peak area resident whose ...,high,compliant,non_renewed,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'My home was completely rebui...,"You are Harriet, a Grizzly Peak resident who r...",[Insurance non-renewal despite full EMBER comp...
1,greg,Leonard,Leonard is a Berkeley city official closely in...,None,None,None,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'I learned that Fire Chief Da...,"You are Leonard, a city official deeply embedd...",[Severe resource constraints limiting fire saf...
2,heidi,Margaret,Margaret is a Berkeley Hills resident who shif...,high,partially_compliant,retained,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'I watched the Northern Calif...,"You are Margaret, a Berkeley Hills resident wh...",[Conflicting advice between Fire Department an...
3,katarina,Evelyn,Evelyn and her spouse are retired homeowners w...,high,compliant,retained,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'I moved to the Berkeley Hill...,"You are Evelyn, a retired homeowner who has li...",[Maintaining insurance coverage in an increasi...
4,matt,Russell,Russell is a Berkeley Hills homeowner who has ...,None,None,None,{'insurance_non_renewal': {'real_response': No...,[{'description': 'I needed to get my internet ...,"You are Russell, a Berkeley Hills homeowner wi...",[Contractor reliability and repeated cancellat...
5,anonymous_1,Jerome,Jerome is a homeowner living in a low-risk fir...,low,None,retained,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'I live in Lafayette in what'...,"You are Jerome, a homeowner in Lafayette livin...",[Eventual loss of insurance coverage despite l...
6,ruth,Claudia,Claudia is a landscape architect and deeply co...,high,compliant,None,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'I helped set an ambitious go...,"You are Claudia, a landscape architect living ...",[Achieving 80% compliance in the high-risk Gri...
7,mrs_shah,Priya,"Priya is a homeowner in Alamo, California, liv...",high,compliant,None,{'insurance_non_renewal': {'real_response': No...,[{'description': 'I undertook a major landscap...,"You are Priya, a homeowner in Alamo, Californi...",[Being sold to or pushed toward products and s...
8,anonymous_1,Walter,Walter is a long-time Berkeley Hills resident ...,high,None,None,{'insurance_non_renewal': {'real_response': '[...,[{'description': 'After the Berkeley fire in t...,"You are Walter, a Berkeley Hills resident with...",[Government has diverted fire safety funds to ...



DISTRIBUTIONS

ID:
ID
anonymous_1    2
candace        1
heidi          1
greg           1
katarina       1
matt           1
ruth           1
mrs_shah       1

Display Name:
Display Name
Harriet     1
Leonard     1
Margaret    1
Evelyn      1
Russell     1
Jerome      1
Claudia     1
Priya       1
Walter      1

Persona:
Persona
Harriet is a Grizzly Peak area resident whose home was rebuilt in 2015 to meet modern California fire codes with ignition-resistant materials and specialized venting. She is fully compliant with all EMBER requirements and the Berkeley Fire Department has cited her property as a 'best-in-class' example of fire risk mitigation. Despite her exemplary efforts, State Farm canceled her fire insurance in 2025, forcing her onto the California FAIR Plan with significantly higher premiums and inadequate coverage. She is proactive and detail-oriented, actively seeking solutions by requesting resource guides to navigate the insurance market and find carriers who recognize 

##Inspect Individual Agent Profile

In [ ]:
# Change this to the agent_id you want to inspect
inspect_agent = "beth"

match = next((a for a in agents if a.get("id") == inspect_agent), None)

if match is None:
    print(f"Agent '{inspect_agent}' not found. Available: {[a.get('id') for a in agents]}")
else:
    print(f"=== {match['display_name'].upper()} ===")
    print(f"\nPERSONA:")
    print(f"  {match['persona']}")

    print(f"\nKEY ATTRIBUTES:")
    print(f"  risk_zone            : {match.get('risk_zone')}")
    print(f"  compliance_status    : {match.get('compliance_status')}")
    print(f"  insurance_status     : {match.get('insurance_status')}")

    print(f"\nHELD OUT RESPONSES:")
    held_out = match.get('held_out_responses', {})
    if held_out.get('insurance_non_renewal'):
        print(f"  Insurance Non-Renewal: ")
        print(f"    Real Response: {held_out['insurance_non_renewal'].get('real_response')}")
        print(f"    Context: {held_out['insurance_non_renewal'].get('context')}")
    if held_out.get('defensible_space_inspection'):
        print(f"  Defensible Space Inspection: ")
        print(f"    Real Response: {held_out['defensible_space_inspection'].get('real_response')}")
        print(f"    Context: {held_out['defensible_space_inspection'].get('context')}")
    if held_out.get('neighbor_pressure'):
        print(f"  Neighbor Pressure: ")
        print(f"    Real Response: {held_out['neighbor_pressure'].get('real_response')}")
        print(f"    Context: {held_out['neighbor_pressure'].get('context')}")
    if held_out.get('firewise_outreach'):
        print(f"  Firewise Outreach: ")
        print(f"    Real Response: {held_out['firewise_outreach'].get('real_response')}")
        print(f"    Context: {held_out['firewise_outreach'].get('context')}")
    if held_out.get('resources_assistance'):
        print(f"  Resources Assistance: ")
        print(f"    Real Response: {held_out['resources_assistance'].get('real_response')}")
        print(f"    Context: {held_out['resources_assistance'].get('context')}")

    print(f"\nSEED MEMORIES:")
    for i, mem in enumerate(match.get('memory_seeds', []), 1):
        print(f"  {i}. {mem.get('description', 'N/A')}")

    print(f"\nSEED NARRATIVE:")
    print(f"  {match.get('seed_narrative')}")


    print(f"\nKEY CONCERNS:")
    for concern in match.get('key_concerns', []):
        print(f"  • {concern}")


Agent 'beth' not found. Available: ['candace', 'greg', 'heidi', 'katarina', 'matt', 'anonymous_1', 'ruth', 'mrs_shah', 'anonymous_1']


## Preview Individual Agent YAML Output


In [ ]:
# Preview the first agent's individual YAML file
if agents:
    first = agents[0]
    aid = first.get('id', 'unknown')
    print(f"--- {aid}.yaml ---")
    print(yaml.dump(first, default_flow_style=False, allow_unicode=True, sort_keys=False))
else:
    print("No agents extracted yet.")


--- candace.yaml ---
id: candace
display_name: Harriet
persona: Harriet is a Grizzly Peak area resident whose home was rebuilt in 2015 to
  meet modern California fire codes with ignition-resistant materials and specialized
  venting. She is fully compliant with all EMBER requirements and the Berkeley Fire
  Department has cited her property as a 'best-in-class' example of fire risk mitigation.
  Despite her exemplary efforts, State Farm canceled her fire insurance in 2025, forcing
  her onto the California FAIR Plan with significantly higher premiums and inadequate
  coverage. She is proactive and detail-oriented, actively seeking solutions by requesting
  resource guides to navigate the insurance market and find carriers who recognize
  high-mitigation efforts.
risk_zone: high
compliance_status: compliant
insurance_status: non_renewed
held_out_responses:
  insurance_non_renewal:
    real_response: '[reconstructed] Even though my property meets all the safety standards
      and BFD c

## Re-run a Single Agent

Use this if one extraction failed or produced a poor result. It re-runs just that one file without re-processing the others.

In [ ]:
# Set to the filename you want to re-run (just the filename, not the full path)
rerun_file = "a020.docx"

rerun_path = notes_dir / rerun_file

if not rerun_path.exists():
    print(f"File not found: {rerun_path}")
else:
    name = rerun_file.split(" - ")[0].split("_")[0].strip()
    print(f"Re-running extraction for: {name}")

    notes_text = extract_text_from_docx(rerun_path)
    profile, raw_response = extract_agent_profile(claude_client, notes_text, name)
    agent_entry = profile_to_agent_entry(profile)

    # Replace in agents list if already exists, otherwise append
    existing_ids = [a.get("id") for a in agents]
    if agent_entry.get("id") in existing_ids:
        idx = existing_ids.index(agent_entry.get("id"))
        agents[idx] = agent_entry
        print(f"Replaced existing entry for '{agent_entry.get('id')}'")
    else:
        agents.append(agent_entry)
        print(f"Added new entry for '{agent_entry.get('id')}'")

    print(f"\nResult:")
    print(json.dumps(agent_entry, indent=2))

    # Save individual YAML for the re-run agent
    agent_id = agent_entry.get("id") or name.lower()
    agent_yaml_path = output_dir / f"{agent_id}.yaml"
    with open(agent_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(agent_entry, f, default_flow_style=False,
                  allow_unicode=True, sort_keys=False)
    print(f"\nSaved → {agent_yaml_path}")


Re-running extraction for: Matt.docx
Replaced existing entry for 'matt'

Result:
{
  "id": "matt",
  "display_name": "Jerome",
  "persona": "Jerome is a Berkeley Hills resident who uses digital platforms and direct approaches to solve home maintenance tasks. He has experience hiring contractors for routine home work, including cable rerouting. His approach to contractors is pragmatic and platform-mediated\u2014he values convenience in discovery and booking but has encountered significant quality and reliability issues. He is self-reliant when work is done poorly, willing to step in and fix problems himself rather than escalate or seek recourse. His experience reflects skepticism about quality assurance in contractor marketplaces, though he continues to use them.",
  "risk_zone": null,
  "compliance_status": null,
  "insurance_status": null,
  "held_out_responses": {
    "insurance_non_renewal": {
      "real_response": null,
      "context": null
    },
    "defensible_space_inspection